# Simulación Roland Garros 2026

Simulación del torneo mediante **Monte Carlo** (10.000 iteraciones) usando el modelo XGBoost entrenado en el notebook anterior para estimar la probabilidad de victoria de cada jugadora en el cuadro.

**Requiere:**
- `src/data/wta_limpio.csv` (generado por notebook 01)
- `src/model/gbx_v3.model` (generado por notebook 02)
- `src/resources/cuadro_python_list.txt` (cuadro del torneo)
- `src/utils/features.py` (funciones auxiliares)

**Arquitectura de la simulación:**

```
┌─────────────────────────────────────────┐
│           SIMULACIÓN MONTE CARLO        │  ← repite 10.000 veces
│                                         │
│   1ª ronda → 2ª → ... → Final          │
│   para cada partido llama a...          │
│                                         │
│   ┌─────────────────────────────────┐   │
│   │         MODELO ML               │   │  ← entrenado con datos históricos
│   │  input: jugadora A vs B         │   │
│   │  output: probabilidad 0..1      │   │
│   └─────────────────────────────────┘   │
│                                         │
└─────────────────────────────────────────┘
         ↓ resultado final
   {"Swiatek I.": 38%, "Sabalenka A.": 22%, ...}
```

## 1. Imports y configuración

In [ ]:
import random
import pickle
import os
import sys
import time
import pandas as pd

# Añadir utils al path para importar features.py
sys.path.append(os.path.join('..', 'utils'))

from features import (forma_reciente, winrate, headtohead, experiencia, get_ranking, get_elo)

In [ ]:
# Rutas relativas al repositorio (ejecutar desde src/notebooks/)
DATA_DIR      = os.path.join('..', 'data')
MODEL_DIR     = os.path.join('..', 'model')
RESOURCES_DIR = os.path.join('..', 'resources')

## 2. Carga de datos y modelo

In [ ]:
# Dataset WTA limpio — fuente de consulta para calcular las features de cada jugadora
wta = pd.read_csv(os.path.join(DATA_DIR, 'wta_limpio.csv'), parse_dates=['Date'])
print(f'Partidos en wta_limpio: {len(wta)}')

In [ ]:
# Modelo de ML (XGBoost — mejor modelo del notebook 02)
with open(os.path.join(MODEL_DIR, 'gbx_v3.model'), 'rb') as f:
    modeloML = pickle.load(f)
print(modeloML)

In [ ]:
# Cuadro del torneo (lista de 128 jugadoras en orden de enfrentamiento)
# El archivo fue generado a partir del cuadro oficial de Roland Garros 2026
# y los nombres se corrigieron manualmente al formato del dataset WTA (Apellido I.)
with open(os.path.join(RESOURCES_DIR, 'cuadro_python_list.txt'), 'r', encoding='utf-8') as f:
    exec(f.read())

print(f'Jugadoras en el cuadro: {len(cuadro)}')
print(cuadro[:10])

## 3. Verificación de nombres

Comprobación de que todos los nombres del cuadro tienen correspondencia en el dataset WTA. Los que no tienen match directo se corrigieron manualmente.

In [ ]:
def verificar_nombres_cuadro(cuadro, wta):
    nombres_wta = set(wta['Player_1'].unique()) | set(wta['Player_2'].unique())
    sin_correspondencia = [j for j in cuadro if j not in nombres_wta]
    return sin_correspondencia

sin_match = verificar_nombres_cuadro(cuadro, wta)
print(f'{len(sin_match)} jugadoras sin correspondencia:')
for j in sin_match:
    print(f'  - {j}')

## 4. Precálculo de features

La simulación repite 10.000 torneos completos. Calcular las features de cada jugadora en cada iteración sería inviable (~1M llamadas). En cambio, se calculan una sola vez antes de la simulación y se almacenan en caché, ya que todas las jugadoras y sus estadísticas son fijas antes del inicio del torneo.

Se calculan también los head-to-head entre todos los pares posibles del cuadro.

In [ ]:
def calculo_features_jugadoras_torneo(df, cuadro):
    """Precalcula y cachea todas las features de las jugadoras del cuadro.
    
    Returns:
        cache_features: dict {jugadora: {feature: valor}}
        cache_h2h:      dict {(p1, p2): h2h_ratio}
    """
    fecha     = pd.to_datetime('2026-05-05')  # Fecha de inicio del torneo
    superficie = 'Clay'
    rondas = ['1st Round', '2nd Round', '3rd Round', '4th Round',
              'Quarterfinals', 'Semifinals', 'The Final']

    cache_features = {}
    for jugadora in cuadro:
        cache_features[jugadora] = {
            'wins2meses':       forma_reciente(df, jugadora, fecha),
            'ratio_superficie': winrate(df, jugadora, fecha, superficie=superficie),
            'experiencia':      experiencia(df, jugadora, fecha),
            'ranking':          get_ranking(df, jugadora, fecha),
            'elo_global':       get_elo(df, jugadora, fecha),
            'elo_superficie':   get_elo(df, jugadora, fecha, superficie),
            'winrate_por_ronda': {
                ronda: winrate(df, jugadora, fecha, ronda=ronda)
                for ronda in rondas
            }
        }

    # H2H entre todos los pares del cuadro
    cache_h2h = {}
    for i, p1 in enumerate(cuadro):
        for p2 in cuadro[i+1:]:
            cache_h2h[(p1, p2)] = headtohead(df, p1, p2, fecha)
            cache_h2h[(p2, p1)] = 1 - cache_h2h[(p1, p2)]

    return cache_features, cache_h2h

In [ ]:
def construir_features_desde_cache(cache_features, cache_h2h, p1, p2, superficie, ronda):
    """Construye el DataFrame de features para un partido a partir de la caché."""
    f1 = cache_features[p1]
    f2 = cache_features[p2]

    row = {
        'surface':             superficie,
        'round':               ronda,
        'rank_diff':           f1['ranking'] - f2['ranking'],
        'wins2meses_p1':       f1['wins2meses'],
        'wins2meses_p2':       f2['wins2meses'],
        'ratio_superficie_p1': f1['ratio_superficie'],
        'ratio_superficie_p2': f2['ratio_superficie'],
        'h2h':                 cache_h2h.get((p1, p2), 0.5),
        'ratio_ronda_p1':      f1['winrate_por_ronda'][ronda],
        'ratio_ronda_p2':      f2['winrate_por_ronda'][ronda],
        'experiencia_p1':      f1['experiencia'],
        'experiencia_p2':      f2['experiencia'],
        'elo_p1':              f1['elo_superficie'],
        'elo_p2':              f2['elo_superficie'],
        'elo_diff':            f1['elo_superficie'] - f2['elo_superficie'],
        'elo_global_p1':       f1['elo_global'],
        'elo_global_p2':       f2['elo_global'],
        'elo_global_diff':     f1['elo_global'] - f2['elo_global'],
        'is_new_p1':           int(f1['experiencia'] < 10),
        'is_new_p2':           int(f2['experiencia'] < 10),
    }
    return pd.DataFrame([row])

In [ ]:
inicio = time.time()
cache_features, cache_h2h = calculo_features_jugadoras_torneo(wta, cuadro)
fin = time.time()
print(f'Caché calculada en {fin - inicio:.1f}s para {len(cuadro)} jugadoras')

## 5. Funciones de simulación

In [ ]:
def simular_partido(prob_a):
    """Simula un partido dado la probabilidad de victoria de la jugadora A.
    Devuelve True si gana A, False si gana B."""
    return random.random() < prob_a


def simular_torneo(cache_features, cache_h2h, cuadro, modelo):
    """Simula un torneo completo desde primera ronda hasta la final.
    
    El cuadro se asume ordenado: posiciones 0-1 enfrentan, 2-3, etc.
    Devuelve el nombre de la campeona.
    """
    superficie = 'Clay'
    rondas = ['1st Round', '2nd Round', '3rd Round', '4th Round',
              'Quarterfinals', 'Semifinals', 'The Final']

    jugadoras   = cuadro.copy()
    indice_ronda = 0

    while len(jugadoras) > 1:
        siguiente_ronda = []
        ronda_actual    = rondas[indice_ronda]

        for i in range(0, len(jugadoras), 2):
            a, b = jugadoras[i], jugadoras[i + 1]
            X    = construir_features_desde_cache(cache_features, cache_h2h, a, b, superficie, ronda_actual)
            prob_a = modelo.predict_proba(X)[0][1]
            ganadora = a if simular_partido(prob_a) else b
            siguiente_ronda.append(ganadora)

        jugadoras    = siguiente_ronda
        indice_ronda += 1

    return jugadoras[0]


def simulacion_montecarlo(cache_features, cache_h2h, cuadro, modelo, n_simulaciones=10000):
    """Ejecuta n_simulaciones torneos completos y devuelve las probabilidades de victoria."""
    victorias = {}

    for _ in range(n_simulaciones):
        campeona = simular_torneo(cache_features, cache_h2h, cuadro, modelo)
        victorias[campeona] = victorias.get(campeona, 0) + 1

    probabilidades = {j: v / n_simulaciones for j, v in victorias.items()}
    return probabilidades, victorias

## 6. Simulación Monte Carlo

In [ ]:
inicio = time.time()
probabilidades, victorias = simulacion_montecarlo(cache_features, cache_h2h, cuadro, modeloML, 10000)
fin = time.time()

print(f'Simulación completada en {fin - inicio:.1f}s')
print()
print('Resultados — Probabilidad de ganar Roland Garros 2026:')
print('-' * 50)
for jugadora, prob in sorted(probabilidades.items(), key=lambda x: x[1], reverse=True):
    barra = '█' * int(prob * 50)
    print(f'{jugadora:<25} {prob:.1%}  {barra}')

In [ ]:
# Mostrar como DataFrame ordenado
resultados_sim = pd.DataFrame([
    {'jugadora': j, 'probabilidad': p, 'victorias_de_10000': victorias[j]}
    for j, p in probabilidades.items()
]).sort_values('probabilidad', ascending=False).reset_index(drop=True)

resultados_sim